In [1]:
import os
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
PROJECT_DIR = '/content/drive/MyDrive/SignLanguageProject'
ZIP_PATH = os.path.join(PROJECT_DIR, 'wlasl-processed.zip')
LOCAL_EXTRACT_DIR = '/content/wlasl_raw'
os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)

# 3. Unzip Dataset Locally (Faster than unzipping on Drive)
print("📦 Unzipping dataset to local Colab storage...")
if not os.path.exists(os.path.join(LOCAL_EXTRACT_DIR, 'videos')):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)
    print("✅ Unzipping complete!")
else:
    print("✅ Dataset already unzipped locally.")

Mounted at /content/drive
📦 Unzipping dataset to local Colab storage...
✅ Unzipping complete!


Metadata Parsing & Top-100 Filtering

In [4]:
import json
import glob
import os

# Ensure PROJECT_DIR is defined
PROJECT_DIR = '/content/drive/MyDrive/SignLanguageProject'
os.makedirs(PROJECT_DIR, exist_ok=True)

# 1. Locate EXACTLY WLASL_v0.3.json
json_matches = glob.glob('/content/wlasl_raw/**/WLASL_v0.3.json', recursive=True)
if not json_matches:
    print("Found these JSONs instead:", glob.glob('/content/wlasl_raw/**/*.json', recursive=True))
    raise FileNotFoundError("Could not locate WLASL_v0.3.json specifically!")

WLASL_JSON_PATH = json_matches[0]
print(f"📄 Found correct JSON metadata at: {WLASL_JSON_PATH}")

# 2. Extract Top 100 Classes
with open(WLASL_JSON_PATH, 'r') as f:
    wlasl_data = json.load(f)

# If JSON loaded as a dict, extract its values into a list
if isinstance(wlasl_data, dict):
    wlasl_data = list(wlasl_data.values())

# Sort by instance count and slice Top 100
sorted_glosses = sorted(wlasl_data, key=lambda x: len(x['instances']), reverse=True)[:100]

class_mapping = {}
target_video_ids = set()

for idx, entry in enumerate(sorted_glosses):
    gloss = entry['gloss']
    class_mapping[gloss] = idx
    for inst in entry['instances']:
        target_video_ids.add(str(inst['video_id']))

# Save mapping for future evaluation
MAPPING_SAVE_PATH = os.path.join(PROJECT_DIR, 'class_mapping_100.json')
with open(MAPPING_SAVE_PATH, 'w') as f:
    json.dump(class_mapping, f, indent=4)

# 3. Filter Physical MP4 Files
all_mp4s = glob.glob('/content/wlasl_raw/**/*.mp4', recursive=True)
selected_videos = [v for v in all_mp4s if os.path.splitext(os.path.basename(v))[0] in target_video_ids]

print(f"🎯 Targeted Top 100 Classes.")
print(f"💾 Class mapping saved to: {MAPPING_SAVE_PATH}")
print(f"🎬 Found {len(selected_videos)} matching .mp4 files on disk for extraction.")

📄 Found correct JSON metadata at: /content/wlasl_raw/WLASL_v0.3.json
🎯 Targeted Top 100 Classes.
💾 Class mapping saved to: /content/drive/MyDrive/SignLanguageProject/class_mapping_100.json
🎬 Found 1013 matching .mp4 files on disk for extraction.


MediaPipe Landmark Extraction

In [5]:
!pip install -q mediapipe opencv-python tqdm

import cv2
import numpy as np
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm

LANDMARKS_DIR = os.path.join(PROJECT_DIR, 'raw_landmarks_100')
os.makedirs(LANDMARKS_DIR, exist_ok=True)

# 1. Download Task Models
if not os.path.exists('pose_landmarker.task'):
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task", 'pose_landmarker.task')
if not os.path.exists('hand_landmarker.task'):
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task", 'hand_landmarker.task')

# 2. Initialize MediaPipe Detectors
pose_detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='pose_landmarker.task', delegate=python.BaseOptions.Delegate.CPU)
))
hand_detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='hand_landmarker.task', delegate=python.BaseOptions.Delegate.CPU), num_hands=2
))

def extract_keypoints(pose_res, hand_res):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in pose_res.pose_landmarks[0]]).flatten() if pose_res.pose_landmarks else np.zeros(33 * 4)
    lh, rh = np.zeros(21 * 3), np.zeros(21 * 3)
    if hand_res.hand_landmarks:
        for idx, hand_info in enumerate(hand_res.handedness):
            landmarks = np.array([[res.x, res.y, res.z] for res in hand_res.hand_landmarks[idx]]).flatten()
            if hand_info[0].category_name == 'Left': lh = landmarks
            elif hand_info[0].category_name == 'Right': rh = landmarks
    return np.concatenate([pose, lh, rh])

# 3. Batch Processing
for video_path in tqdm(selected_videos, desc="Extracting Landmarks"):
    vid_id = os.path.splitext(os.path.basename(video_path))[0]
    save_path = os.path.join(LANDMARKS_DIR, f"{vid_id}.npy")

    if os.path.exists(save_path): continue

    cap = cv2.VideoCapture(video_path)
    frames_data = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        frames_data.append(extract_keypoints(pose_detector.detect(mp_image), hand_detector.detect(mp_image)))

    cap.release()
    if frames_data: np.save(save_path, np.array(frames_data))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.4 MB/s eta 0:00:00


Extracting Landmarks: 100%|██████████| 1013/1013 [2:28:47<00:00,  8.81s/it]


In [6]:
import os
import shutil
import glob
from tqdm import tqdm

# Define directories
PROJECT_DIR = '/content/drive/MyDrive/SignLanguageProject'
SOURCE_DIR = os.path.join(PROJECT_DIR, 'raw_landmarks_100')
DEST_DIR = os.path.join(PROJECT_DIR, 'new landmark wlasl 100')

os.makedirs(DEST_DIR, exist_ok=True)

# Find all extracted .npy files
extracted_files = glob.glob(os.path.join(SOURCE_DIR, '*.npy'))
print(f"Found {len(extracted_files)} files in '{SOURCE_DIR}'.")

# Copy files to the new directory
for file_path in tqdm(extracted_files, desc="Copying landmarks to new folder"):
    filename = os.path.basename(file_path)
    dest_path = os.path.join(DEST_DIR, filename)
    if not os.path.exists(dest_path):
        shutil.copy(file_path, dest_path)

print(f"\n✅ Successfully saved all {len(extracted_files)} files to: {DEST_DIR}")

Found 1013 files in '/content/drive/MyDrive/SignLanguageProject/raw_landmarks_100'.


Copying landmarks to new folder: 100%|██████████| 1013/1013 [00:26<00:00, 38.55it/s]


✅ Successfully saved all 1013 files to: /content/drive/MyDrive/SignLanguageProject/new landmark wlasl 100


In [7]:
import pandas as pd
import glob

PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_landmarks_100')
os.makedirs(PROCESSED_DIR, exist_ok=True)

raw_files = glob.glob(os.path.join(LANDMARKS_DIR, '*.npy'))

def process_sequence(seq):
    # 1. Linear Interpolation for internal missing hands
    lh_missing = np.sum(np.abs(seq[:, 132:195]), axis=1) == 0
    rh_missing = np.sum(np.abs(seq[:, 195:258]), axis=1) == 0
    seq[lh_missing, 132:195] = np.nan
    seq[rh_missing, 195:258] = np.nan
    seq = pd.DataFrame(seq).interpolate(method='linear', limit_area='inside').fillna(0.0).to_numpy()

    # 2. Spatial Normalization (Anchor to Nose [0], Scale by Shoulders [11, 12])
    all_x = np.concatenate([np.arange(0, 132, 4), np.arange(132, 195, 3), np.arange(195, 258, 3)])
    all_y = np.concatenate([np.arange(1, 132, 4), np.arange(133, 195, 3), np.arange(196, 258, 3)])
    all_z = np.concatenate([np.arange(2, 132, 4), np.arange(134, 195, 3), np.arange(197, 258, 3)])

    for i in range(len(seq)):
        nose_x, nose_y, nose_z = seq[i, 0], seq[i, 1], seq[i, 2]
        shoulder_dist = np.sqrt((seq[i, 44] - seq[i, 48])**2 + (seq[i, 45] - seq[i, 49])**2)
        if shoulder_dist == 0: shoulder_dist = 1.0

        seq[i, all_x] = (seq[i, all_x] - nose_x) / shoulder_dist
        seq[i, all_y] = (seq[i, all_y] - nose_y) / shoulder_dist
        seq[i, all_z] = (seq[i, all_z] - nose_z) / shoulder_dist

    # 3. Add Velocity Features (Frame-to-Frame Derivative)
    velocity = np.zeros_like(seq)
    velocity[1:] = seq[1:] - seq[:-1]

    return np.concatenate([seq, velocity], axis=-1)  # Shape: (T, 516)

for file_path in tqdm(raw_files, desc="Feature Engineering"):
    save_path = os.path.join(PROCESSED_DIR, os.path.basename(file_path))
    if os.path.exists(save_path):
        continue
    try:
        data = np.load(file_path)
        if data.shape[0] < 15: continue # Drop corrupted or ultra-short clips
        processed_data = process_sequence(data.astype(np.float32))
        np.save(save_path, processed_data)
    except Exception as e:
        print(f"Skipping {file_path} due to error: {e}")

print(f"\n✅ Preprocessing Complete! Clean arrays saved to {PROCESSED_DIR}")

Feature Engineering: 100%|██████████| 1013/1013 [00:27<00:00, 37.09it/s]


✅ Preprocessing Complete! Clean arrays saved to /content/drive/MyDrive/SignLanguageProject/processed_landmarks_100


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

# --- 1. DATA PREPARATION WITH GROUP SHUFFLE SPLIT ---
all_files = glob.glob(os.path.join(PROCESSED_DIR, '*.npy'))
file_list, label_list, base_ids = [], [], []

# Re-map video IDs to labels
with open(os.path.join(PROJECT_DIR, 'class_mapping_100.json'), 'r') as f:
    class_mapping = json.load(f)

video_to_class = {}
with open(WLASL_JSON_PATH, 'r') as f:
    wlasl_data = json.load(f)
    if isinstance(wlasl_data, dict): wlasl_data = list(wlasl_data.values())
    for entry in wlasl_data:
        gloss = entry['gloss']
        if gloss in class_mapping:
            for inst in entry['instances']:
                video_to_class[str(inst['video_id'])] = class_mapping[gloss]

for f in all_files:
    vid_id = os.path.basename(f).replace('.npy', '').replace('_aug', '')
    if vid_id in video_to_class:
        file_list.append(f)
        label_list.append(video_to_class[vid_id])
        base_ids.append(vid_id)

# 80/10/10 Split without Leakage
gss_train = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss_train.split(file_list, label_list, groups=base_ids))

temp_files = [file_list[i] for i in temp_idx]
temp_labels = [label_list[i] for i in temp_idx]
temp_groups = [base_ids[i] for i in temp_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_files, temp_labels, groups=temp_groups))

def get_split(indices, source_files, source_labels):
    return [source_files[i] for i in indices], [source_labels[i] for i in indices]

train_f, train_l = get_split(train_idx, file_list, label_list)
val_f, val_l = get_split(val_idx, temp_files, temp_labels)
test_f, test_l = get_split(test_idx, temp_files, temp_labels)

# --- 2. FAST IN-MEMORY DATASET ---
class SignDataset(Dataset):
    def __init__(self, files, labels, max_frames=40):
        self.labels = labels
        print(f"Preloading {len(files)} files into RAM...")
        self.data = [np.load(f).astype(np.float32) for f in files]
        self.max_frames = max_frames

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        seq = self.data[idx]
        num_frames, feat_dim = seq.shape
        padded = np.zeros((self.max_frames, feat_dim), dtype=np.float32)

        if num_frames > self.max_frames:
            start = (num_frames - self.max_frames) // 2
            padded[:] = seq[start:start+self.max_frames, :]
        else:
            padded[:num_frames, :] = seq

        return torch.tensor(padded), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(SignDataset(train_f, train_l), batch_size=32, shuffle=True)
val_loader = DataLoader(SignDataset(val_f, val_l), batch_size=32)
test_loader = DataLoader(SignDataset(test_f, test_l), batch_size=32)

# --- 3. MODEL ARCHITECTURE ---
class ConvBiLSTM(nn.Module):
    def __init__(self, input_dim=516, hidden_dim=128, num_classes=100):
        super(ConvBiLSTM, self).__init__()
        self.conv1d = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.lstm = nn.LSTM(128, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1d(x).transpose(1, 2)
        lstm_out, _ = self.lstm(x)
        pooled = torch.cat([torch.mean(lstm_out, dim=1), torch.max(lstm_out, dim=1)[0]], dim=1)
        return self.fc(pooled)

# --- 4. TRAINING LOOP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConvBiLSTM().to(device)

# Handle Class Imbalance & Overfitting
class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_l), y=train_l)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_wts, dtype=torch.float32).to(device), label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-5)

best_val = 0.0
for epoch in range(80):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels.to(device)).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs.to(device))
            val_correct += (outputs.argmax(1) == labels.to(device)).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), os.path.join(PROJECT_DIR, 'best_model.pth'))
        save_msg = "⭐ [BEST]"
    else:
        save_msg = ""

    print(f"Epoch {epoch+1:02d}/80 | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% {save_msg}")

print(f"\n🎉 Training Complete! Best Validation Accuracy: {best_val*100:.2f}%")

Preloading 810 files into RAM...
Preloading 101 files into RAM...
Preloading 102 files into RAM...
Epoch 01/80 | Train Acc: 1.23% | Val Acc: 2.97% ⭐ [BEST]
Epoch 02/80 | Train Acc: 3.70% | Val Acc: 2.97% 
Epoch 03/80 | Train Acc: 7.28% | Val Acc: 3.96% ⭐ [BEST]
Epoch 04/80 | Train Acc: 10.00% | Val Acc: 6.93% ⭐ [BEST]
Epoch 05/80 | Train Acc: 14.07% | Val Acc: 6.93% 
Epoch 06/80 | Train Acc: 16.54% | Val Acc: 9.90% ⭐ [BEST]
Epoch 07/80 | Train Acc: 19.01% | Val Acc: 10.89% ⭐ [BEST]
Epoch 08/80 | Train Acc: 21.98% | Val Acc: 15.84% ⭐ [BEST]
Epoch 09/80 | Train Acc: 27.28% | Val Acc: 13.86% 
Epoch 10/80 | Train Acc: 29.51% | Val Acc: 10.89% 
Epoch 11/80 | Train Acc: 31.36% | Val Acc: 11.88% 
Epoch 12/80 | Train Acc: 36.67% | Val Acc: 15.84% 
Epoch 13/80 | Train Acc: 39.75% | Val Acc: 19.80% ⭐ [BEST]
Epoch 14/80 | Train Acc: 44.94% | Val Acc: 16.83% 
Epoch 15/80 | Train Acc: 44.44% | Val Acc: 13.86% 
Epoch 16/80 | Train Acc: 47.16% | Val Acc: 26.73% ⭐ [BEST]
Epoch 17/80 | Train Acc: 53.58

In [13]:
import torch
import torch.nn as nn
import numpy as np

class FullyConnectedGraphConv(nn.Module):
    def __init__(self, in_features, out_features, num_nodes=258):
        super(FullyConnectedGraphConv, self).__init__()
        # Trainable adjacency matrix A in R^{K x K}
        self.A = nn.Parameter(torch.randn(num_nodes, num_nodes) * 0.01)
        # Trainable weight matrix W in R^{F x F'}
        self.W = nn.Linear(in_features, out_features)

    def forward(self, x):
        # x shape: (Batch, Time, Nodes, Features)

        # 1. Spatial Graph Mixing: A * H
        # Multiplies the adjacency matrix across the Node dimension
        x = torch.einsum('vw, btwc -> btvc', self.A, x)

        # 2. Feature Mixing: H * W
        x = self.W(x)

        # 3. Activation: tanh (as specified in the paper)
        return torch.tanh(x)


class ResidualGraphBlock(nn.Module):
    def __init__(self, in_features, out_features, num_nodes=258):
        super(ResidualGraphBlock, self).__init__()
        # Stacks two graph convolutional layers per block
        self.gcn1 = FullyConnectedGraphConv(in_features, out_features, num_nodes)
        self.gcn2 = FullyConnectedGraphConv(out_features, out_features, num_nodes)

        # Residual connection
        if in_features != out_features:
            self.res = nn.Linear(in_features, out_features)
        else:
            self.res = nn.Identity()

    def forward(self, x):
        res = self.res(x)
        out = self.gcn1(x)
        out = self.gcn2(out)
        return out + res


class PoseTGCN(nn.Module):
    def __init__(self, num_nodes=258, in_features=2, num_classes=100):
        super(PoseTGCN, self).__init__()

        # Stacking multiple residual graph convolutional blocks
        self.block1 = ResidualGraphBlock(in_features, 32, num_nodes)
        self.block2 = ResidualGraphBlock(32, 64, num_nodes)
        self.block3 = ResidualGraphBlock(64, 128, num_nodes)

        # Regularization before classification
        self.dropout = nn.Dropout(0.4)

        # Final classification head
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input x shape: (Batch, Time, 516)
        B, T, D = x.shape

        # SPLIT AND RE-STACK: Convert 516 flat features into (258 Nodes, 2 Channels)
        pos = x[:, :, :258]
        vel = x[:, :, 258:]

        # New shape: (Batch, Time, 258 Nodes, 2 Features)
        x_graph = torch.stack([pos, vel], dim=-1)

        # Pass through the Graph Convolutional Blocks
        out = self.block1(x_graph)
        out = self.block2(out)
        out = self.block3(out) # Shape: (B, T, 258, 128)

        # Average pooling along the temporal dimension
        out = torch.mean(out, dim=1) # Shape: (B, 258, 128)

        # Average pooling along the spatial (node) dimension
        out = torch.mean(out, dim=1) # Shape: (B, 128)

        out = self.dropout(out)
        return self.fc(out)

In [14]:
# Initialize the TGCN model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PoseTGCN().to(device)

# Use balanced class weights
class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_l), y=train_l)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_wts, dtype=torch.float32).to(device), label_smoothing=0.1)

# Lower the learning rate for Graph stability
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

# (Proceed with your standard training loop as normal)

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# --- 1. TOP-K ACCURACY HELPER ---
def accuracy_top_k(outputs, targets, topk=(1, 5)):
    """Computes Top-1 and Top-5 accuracy for a given batch."""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = targets.size(0)

        # Get the top K predictions
        _, pred = outputs.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.item() / batch_size)
        return res

# --- 2. TRAINING PREPARATION ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PoseTGCN().to(device) # Using the TGCN model block from the previous step

# Compute balanced class weights to handle dataset imbalance
class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_l), y=train_l)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_wts, dtype=torch.float32).to(device), label_smoothing=0.1)

# Lower learning rate for Graph Convolutional stability
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

best_val_top1 = 0.0

# --- 3. THE TRAINING LOOP ---
for epoch in range(100):
    model.train()
    running_loss = 0.0
    train_correct_top1, train_total = 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping prevents exploding gradients in graph networks
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        # Track Train Top-1
        preds_top1 = outputs.argmax(1)
        train_correct_top1 += (preds_top1 == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct_top1 / train_total
    scheduler.step()

    # --- VALIDATION PHASE ---
    model.eval()
    val_top1_accs, val_top5_accs = [], []
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            # Calculate Top-1 and Top-5 for the batch
            top1, top5 = accuracy_top_k(outputs, labels, topk=(1, 5))

            # Weight by batch size to get accurate epoch averages
            batch_size = labels.size(0)
            val_top1_accs.append(top1 * batch_size)
            val_top5_accs.append(top5 * batch_size)
            val_total += batch_size

    epoch_val_top1 = sum(val_top1_accs) / val_total
    epoch_val_top5 = sum(val_top5_accs) / val_total

    # Save Best Model based on Validation Top-1
    if epoch_val_top1 > best_val_top1:
        best_val_top1 = epoch_val_top1
        torch.save(model.state_dict(), os.path.join(PROJECT_DIR, 'best_tgcn_model.pth'))
        save_msg = "⭐ [BEST]"
    else:
        save_msg = ""

    print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc*100:.2f}% | Val Top-1: {epoch_val_top1*100:.2f}% | Val Top-5: {epoch_val_top5*100:.2f}% {save_msg}")


# --- 4. FINAL TEST EVALUATION ---
print("\n" + "=" * 50)
print("🏆 RUNNING FINAL TEST EVALUATION ON UNSEEN DATA")
print("=" * 50)

# Load the best saved weights
model.load_state_dict(torch.load(os.path.join(PROJECT_DIR, 'best_tgcn_model.pth')))
model.eval()

test_top1_accs, test_top5_accs = [], []
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        top1, top5 = accuracy_top_k(outputs, labels, topk=(1, 5))
        batch_size = labels.size(0)

        test_top1_accs.append(top1 * batch_size)
        test_top5_accs.append(top5 * batch_size)
        test_total += batch_size

final_test_top1 = sum(test_top1_accs) / test_total
final_test_top5 = sum(test_top5_accs) / test_total

print(f"🌟 FINAL TEST TOP-1 ACCURACY : {final_test_top1*100:.2f}%")
print(f"🌟 FINAL TEST TOP-5 ACCURACY : {final_test_top5*100:.2f}%")
print("=" * 50)

Epoch 001/100 | Train Acc: 0.25% | Val Top-1: 1.98% | Val Top-5: 2.97% ⭐ [BEST]
Epoch 002/100 | Train Acc: 0.99% | Val Top-1: 0.99% | Val Top-5: 8.91% 
Epoch 003/100 | Train Acc: 1.36% | Val Top-1: 3.96% | Val Top-5: 8.91% ⭐ [BEST]
Epoch 004/100 | Train Acc: 1.48% | Val Top-1: 0.99% | Val Top-5: 7.92% 
Epoch 005/100 | Train Acc: 1.48% | Val Top-1: 1.98% | Val Top-5: 10.89% 
Epoch 006/100 | Train Acc: 1.73% | Val Top-1: 1.98% | Val Top-5: 8.91% 
Epoch 007/100 | Train Acc: 2.22% | Val Top-1: 0.99% | Val Top-5: 9.90% 
Epoch 008/100 | Train Acc: 1.85% | Val Top-1: 2.97% | Val Top-5: 9.90% 
Epoch 009/100 | Train Acc: 2.22% | Val Top-1: 3.96% | Val Top-5: 14.85% 
Epoch 010/100 | Train Acc: 3.95% | Val Top-1: 0.99% | Val Top-5: 13.86% 
Epoch 011/100 | Train Acc: 3.70% | Val Top-1: 3.96% | Val Top-5: 13.86% 
Epoch 012/100 | Train Acc: 3.58% | Val Top-1: 5.94% | Val Top-5: 14.85% ⭐ [BEST]
Epoch 013/100 | Train Acc: 4.44% | Val Top-1: 7.92% | Val Top-5: 22.77% ⭐ [BEST]
Epoch 014/100 | Train Acc:

KeyboardInterrupt: 

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

# --- 1. CLEAN DATASET (NO DESTRUCTIVE NORMALIZATION) ---
class CleanSequenceDataset(Dataset):
    def __init__(self, files, labels, max_frames=40, is_train=True):
        self.labels = labels
        self.max_frames = max_frames
        self.is_train = is_train

        # Preload raw landmarks into RAM
        self.data = [np.load(f).astype(np.float32) for f in files]

    def __len__(self):
        return len(self.labels)

    def _augment(self, seq):
        # Apply scaling ONLY to valid non-zero keypoints
        mask = (seq != 0.0).astype(np.float32)
        scale = np.random.uniform(0.92, 1.08)
        seq = (seq * scale) + (np.random.normal(0, 0.003, size=seq.shape).astype(np.float32) * mask)
        return seq * mask

    def __getitem__(self, idx):
        seq = self.data[idx].copy()

        if self.is_train:
            seq = self._augment(seq)

        num_frames, feat_dim = seq.shape
        padded = np.zeros((self.max_frames, feat_dim), dtype=np.float32)

        # Center padding
        if num_frames > self.max_frames:
            start = (num_frames - self.max_frames) // 2
            padded[:] = seq[start:start+self.max_frames, :]
        else:
            start = (self.max_frames - num_frames) // 2
            padded[start:start+num_frames, :] = seq

        return torch.tensor(padded, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(CleanSequenceDataset(train_f, train_l, is_train=True), batch_size=32, shuffle=True)
val_loader = DataLoader(CleanSequenceDataset(val_f, val_l, is_train=False), batch_size=32)
test_loader = DataLoader(CleanSequenceDataset(test_f, test_l, is_train=False), batch_size=32)


# --- 2. HIGH-PERFORMANCE CONV1D + BILSTM ARCHITECTURE ---
class RobustConvBiLSTM(nn.Module):
    def __init__(self, input_dim=516, hidden_dim=128, num_classes=100):
        super(RobustConvBiLSTM, self).__init__()

        # 1D Temporal Convolution
        self.conv1d = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # 2-Layer BiLSTM
        self.lstm = nn.LSTM(128, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)

        # Dual Temporal Pooling (Mean + Max)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, T, D)
        x_conv = x.transpose(1, 2)
        x_conv = self.conv1d(x_conv).transpose(1, 2)

        lstm_out, _ = self.lstm(x_conv)

        # Concatenate Average and Max pooling across time
        avg_pool = torch.mean(lstm_out, dim=1)
        max_pool = torch.max(lstm_out, dim=1)[0]
        pooled = torch.cat([avg_pool, max_pool], dim=1)

        return self.fc(pooled)


# --- 3. STABLE TRAINING LOOP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RobustConvBiLSTM().to(device)

class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_l), y=train_l)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_wts, dtype=torch.float32).to(device), label_smoothing=0.1)

# Standard AdamW
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

best_val_top1 = 0.0

for epoch in range(100):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    scheduler.step()

    model.eval()
    val_top1_correct, val_top5_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            # Top-1
            preds_top1 = outputs.argmax(1)
            val_top1_correct += (preds_top1 == labels).sum().item()

            # Top-5
            _, preds_top5 = outputs.topk(5, dim=1)
            for i in range(labels.size(0)):
                if labels[i] in preds_top5[i]:
                    val_top5_correct += 1

            val_total += labels.size(0)

    val_top1 = val_top1_correct / val_total
    val_top5 = val_top5_correct / val_total

    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        torch.save(model.state_dict(), os.path.join(PROJECT_DIR, 'best_convbilstm_model.pth'))
        save_msg = "⭐ [BEST]"
    else:
        save_msg = ""

    print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc*100:.2f}% | Val Top-1: {val_top1*100:.2f}% | Val Top-5: {val_top5*100:.2f}% {save_msg}")

print(f"\n🎉 Training Complete! Best Validation Top-1 Accuracy: {best_val_top1*100:.2f}%")

Epoch 001/100 | Train Acc: 1.98% | Val Top-1: 1.98% | Val Top-5: 8.91% ⭐ [BEST]
Epoch 002/100 | Train Acc: 6.54% | Val Top-1: 4.95% | Val Top-5: 11.88% ⭐ [BEST]
Epoch 003/100 | Train Acc: 5.80% | Val Top-1: 2.97% | Val Top-5: 19.80% 
Epoch 004/100 | Train Acc: 11.98% | Val Top-1: 6.93% | Val Top-5: 32.67% ⭐ [BEST]
Epoch 005/100 | Train Acc: 13.09% | Val Top-1: 10.89% | Val Top-5: 30.69% ⭐ [BEST]
Epoch 006/100 | Train Acc: 14.57% | Val Top-1: 8.91% | Val Top-5: 32.67% 
Epoch 007/100 | Train Acc: 20.12% | Val Top-1: 8.91% | Val Top-5: 39.60% 
Epoch 008/100 | Train Acc: 23.09% | Val Top-1: 9.90% | Val Top-5: 30.69% 
Epoch 009/100 | Train Acc: 24.81% | Val Top-1: 12.87% | Val Top-5: 36.63% ⭐ [BEST]
Epoch 010/100 | Train Acc: 26.54% | Val Top-1: 7.92% | Val Top-5: 45.54% 
Epoch 011/100 | Train Acc: 32.35% | Val Top-1: 16.83% | Val Top-5: 41.58% ⭐ [BEST]
Epoch 012/100 | Train Acc: 35.31% | Val Top-1: 9.90% | Val Top-5: 34.65% 
Epoch 013/100 | Train Acc: 40.74% | Val Top-1: 15.84% | Val Top-5